# WC_BADGE_PRODUCT_D ETL - ODI to Databricks Migration

**Target Table:** `workspace.prxbi_dw_sep.WC_BADGE_PRODUCT_D`

**Source Table:** `workspace.prxbi_ts_sep.WC_MERCURY_PRODUCT_TS`

**ETL Process:** Dimension Load with NOT EXISTS Change Detection

**ODI Package:** WC_BADGE_PRODUCT_D Load

---

## Step 1: Variables as TEMP VIEWs

In [ ]:
%sql
-- V_ETL_LAST_EXTRACT_TIME: mapped from #GLOBAL.v_ETL_JOB_TYPE
CREATE OR REPLACE TEMPORARY VIEW VAR_ETL_LAST_EXTRACT_TIME AS
SELECT etl_last_extract_time
FROM workspace.prxbi_dw_sep.wc_etl_parameters
WHERE ETL_JOB_TYPE = (SELECT * FROM VAR_V_ETL_JOB_TYPE)

In [ ]:
%sql
-- V_ETL_CURRENT_EXTRACT_TIME: mapped from #GLOBAL.v_ETL_JOB_TYPE
CREATE OR REPLACE TEMPORARY VIEW VAR_ETL_CURRENT_EXTRACT_TIME AS
SELECT etl_current_extract_time
FROM workspace.prxbi_dw_sep.wc_etl_parameters
WHERE ETL_JOB_TYPE = (SELECT * FROM VAR_V_ETL_JOB_TYPE)

In [ ]:
%sql
-- V_ETL_PROC_WID: mapped from #GLOBAL.v_ETL_JOB_TYPE
CREATE OR REPLACE TEMPORARY VIEW VAR_ETL_PROC_WID AS
SELECT ROW_WID
FROM workspace.prxbi_dw_sep.wc_etl_parameters
WHERE ETL_JOB_TYPE = (SELECT * FROM VAR_V_ETL_JOB_TYPE)

---

## Step 2: C$ Work Table as TEMP VIEW (Source Extract with Deduplication)

In [ ]:
%sql
-- C$_0FILTER: Extract from WC_MERCURY_PRODUCT_TS with ROW_NUMBER dedup
CREATE OR REPLACE TEMPORARY VIEW C_0FILTER AS
SELECT
    DISTINCT_PRODUCT_D.NAME,
    DISTINCT_PRODUCT_D.PRICE,
    DISTINCT_PRODUCT_D.STATUS,
    DISTINCT_PRODUCT_D.VISIBILITY,
    DISTINCT_PRODUCT_D.WEIGHT,
    DISTINCT_PRODUCT_D.ID,
    DISTINCT_PRODUCT_D.PARENT_SKU,
    DISTINCT_PRODUCT_D.PARENT_RX_EBS_PRODUCT_CODE,
    DISTINCT_PRODUCT_D.CREATED_AT,
    DISTINCT_PRODUCT_D.UPDATED_AT,
    DISTINCT_PRODUCT_D.STOCK_QTY,
    DISTINCT_PRODUCT_D.RX_LINK_TYPE
FROM (
    SELECT DISTINCT
        WC_MERCURY_PRODUCT_TS.NAME,
        WC_MERCURY_PRODUCT_TS.PARENT_PRICE AS PRICE,
        WC_MERCURY_PRODUCT_TS.STATUS,
        WC_MERCURY_PRODUCT_TS.VISIBILITY,
        WC_MERCURY_PRODUCT_TS.WEIGHT,
        WC_MERCURY_PRODUCT_TS.INT_INSERT_DATE,
        WC_MERCURY_PRODUCT_TS.INT_UPDATE_DATE,
        ROW_NUMBER() OVER (
            PARTITION BY WC_MERCURY_PRODUCT_TS.ID
            ORDER BY WC_MERCURY_PRODUCT_TS.INT_INSERT_DATE DESC
        ) AS RNK,
        WC_MERCURY_PRODUCT_TS.ID,
        WC_MERCURY_PRODUCT_TS.SKU,
        WC_MERCURY_PRODUCT_TS.PARENT_SKU,
        WC_MERCURY_PRODUCT_TS.PARENT_PRICE,
        WC_MERCURY_PRODUCT_TS.PARENT_RX_EBS_PRODUCT_CODE,
        WC_MERCURY_PRODUCT_TS.CREATED_AT,
        WC_MERCURY_PRODUCT_TS.UPDATED_AT,
        WC_MERCURY_PRODUCT_TS.STOCK_QTY,
        WC_MERCURY_PRODUCT_TS.RX_LINK_TYPE
    FROM workspace.prxbi_ts_sep.WC_MERCURY_PRODUCT_TS
) DISTINCT_PRODUCT_D
WHERE DISTINCT_PRODUCT_D.RNK = 1

---

## Step 3: I$ Flow Table as TEMP VIEW (Transformations + Change Detection)

In [ ]:
%sql
-- I$_WC_BADGE_PRODUCT_D: Column mapping + NOT EXISTS change detection
-- Uses LEFT JOIN approach: target not found = 'I' (insert), target found but columns differ = 'U' (update)
CREATE OR REPLACE TEMPORARY VIEW I_WC_BADGE_PRODUCT_D AS
SELECT
    S.ID,
    S.SKU,
    S.NAME,
    S.PRICE,
    S.STATUS,
    S.VISIBILITY,
    S.WEIGHT,
    S.STOCK_QTY,
    S.INTEGRATION_ID,
    S.DATASOURCE_NUM_ID,
    S.W_INSERT_DT,
    S.W_UPDATE_DT,
    S.PARENT_RX_EBS_PRODUCT_CODE,
    S.CHANGED_ON_DT,
    S.CREATED_ON_DT,
    S.RX_LINK_TYPE,
    CASE
        WHEN T.INTEGRATION_ID IS NULL THEN 'I'
        ELSE 'U'
    END AS IND_UPDATE
FROM (
    SELECT
        FILTER_A.ID AS ID,
        FILTER_A.PARENT_SKU AS SKU,
        FILTER_A.NAME AS NAME,
        FILTER_A.PRICE AS PRICE,
        FILTER_A.STATUS AS STATUS,
        FILTER_A.VISIBILITY AS VISIBILITY,
        FILTER_A.WEIGHT AS WEIGHT,
        FILTER_A.STOCK_QTY AS STOCK_QTY,
        FILTER_A.ID AS INTEGRATION_ID,
        380 AS DATASOURCE_NUM_ID,
        current_timestamp() AS W_INSERT_DT,
        current_timestamp() AS W_UPDATE_DT,
        FILTER_A.PARENT_RX_EBS_PRODUCT_CODE AS PARENT_RX_EBS_PRODUCT_CODE,
        FILTER_A.UPDATED_AT AS CHANGED_ON_DT,
        FILTER_A.CREATED_AT AS CREATED_ON_DT,
        FILTER_A.RX_LINK_TYPE AS RX_LINK_TYPE
    FROM C_0FILTER FILTER_A
) S
LEFT JOIN workspace.prxbi_dw_sep.WC_BADGE_PRODUCT_D T
    ON T.INTEGRATION_ID = S.INTEGRATION_ID
    AND T.DATASOURCE_NUM_ID = S.DATASOURCE_NUM_ID
WHERE T.INTEGRATION_ID IS NULL
   OR NOT (
        ((T.ID = S.ID) OR (T.ID IS NULL AND S.ID IS NULL))
        AND ((T.SKU = S.SKU) OR (T.SKU IS NULL AND S.SKU IS NULL))
        AND ((T.NAME = S.NAME) OR (T.NAME IS NULL AND S.NAME IS NULL))
        AND ((T.PRICE = S.PRICE) OR (T.PRICE IS NULL AND S.PRICE IS NULL))
        AND ((T.STATUS = S.STATUS) OR (T.STATUS IS NULL AND S.STATUS IS NULL))
        AND ((T.VISIBILITY = S.VISIBILITY) OR (T.VISIBILITY IS NULL AND S.VISIBILITY IS NULL))
        AND ((T.WEIGHT = S.WEIGHT) OR (T.WEIGHT IS NULL AND S.WEIGHT IS NULL))
        AND ((T.STOCK_QTY = S.STOCK_QTY) OR (T.STOCK_QTY IS NULL AND S.STOCK_QTY IS NULL))
        AND ((T.W_UPDATE_DT = S.W_UPDATE_DT) OR (T.W_UPDATE_DT IS NULL AND S.W_UPDATE_DT IS NULL))
        AND ((T.PARENT_RX_EBS_PRODUCT_CODE = S.PARENT_RX_EBS_PRODUCT_CODE) OR (T.PARENT_RX_EBS_PRODUCT_CODE IS NULL AND S.PARENT_RX_EBS_PRODUCT_CODE IS NULL))
        AND ((T.CHANGED_ON_DT = S.CHANGED_ON_DT) OR (T.CHANGED_ON_DT IS NULL AND S.CHANGED_ON_DT IS NULL))
        AND ((T.CREATED_ON_DT = S.CREATED_ON_DT) OR (T.CREATED_ON_DT IS NULL AND S.CREATED_ON_DT IS NULL))
        AND ((T.RX_LINK_TYPE = S.RX_LINK_TYPE) OR (T.RX_LINK_TYPE IS NULL AND S.RX_LINK_TYPE IS NULL))
    )

---

## Step 4: MERGE for Updates (IND_UPDATE = 'U')

In [ ]:
%sql
-- Update existing records in target where columns have changed
MERGE INTO workspace.prxbi_dw_sep.WC_BADGE_PRODUCT_D AS T
USING (
    SELECT * FROM I_WC_BADGE_PRODUCT_D WHERE IND_UPDATE = 'U'
) AS S
ON T.INTEGRATION_ID = S.INTEGRATION_ID
    AND T.DATASOURCE_NUM_ID = S.DATASOURCE_NUM_ID
WHEN MATCHED THEN UPDATE SET
    T.ID = S.ID,
    T.SKU = S.SKU,
    T.NAME = S.NAME,
    T.PRICE = S.PRICE,
    T.STATUS = S.STATUS,
    T.VISIBILITY = S.VISIBILITY,
    T.WEIGHT = S.WEIGHT,
    T.STOCK_QTY = S.STOCK_QTY,
    T.W_UPDATE_DT = current_timestamp(),
    T.PARENT_RX_EBS_PRODUCT_CODE = S.PARENT_RX_EBS_PRODUCT_CODE,
    T.CHANGED_ON_DT = S.CHANGED_ON_DT,
    T.CREATED_ON_DT = S.CREATED_ON_DT,
    T.RX_LINK_TYPE = S.RX_LINK_TYPE,
    T.ETL_PROC_WID = (SELECT ROW_WID FROM VAR_ETL_PROC_WID)

---

## Step 5: INSERT for New Rows (IND_UPDATE = 'I')

In [ ]:
%sql
-- Insert new records into target with generated ROW_WID
INSERT INTO workspace.prxbi_dw_sep.WC_BADGE_PRODUCT_D (
    ROW_WID,
    ID,
    SKU,
    NAME,
    PRICE,
    STATUS,
    VISIBILITY,
    WEIGHT,
    STOCK_QTY,
    INTEGRATION_ID,
    DATASOURCE_NUM_ID,
    W_INSERT_DT,
    W_UPDATE_DT,
    PARENT_RX_EBS_PRODUCT_CODE,
    CHANGED_ON_DT,
    CREATED_ON_DT,
    RX_LINK_TYPE,
    ETL_PROC_WID
)
SELECT
    row_number() OVER (ORDER BY S.INTEGRATION_ID) + COALESCE((SELECT MAX(ROW_WID) FROM workspace.prxbi_dw_sep.WC_BADGE_PRODUCT_D), 0) AS ROW_WID,
    S.ID,
    S.SKU,
    S.NAME,
    S.PRICE,
    S.STATUS,
    S.VISIBILITY,
    S.WEIGHT,
    S.STOCK_QTY,
    S.INTEGRATION_ID,
    S.DATASOURCE_NUM_ID,
    current_timestamp() AS W_INSERT_DT,
    current_timestamp() AS W_UPDATE_DT,
    S.PARENT_RX_EBS_PRODUCT_CODE,
    S.CHANGED_ON_DT,
    S.CREATED_ON_DT,
    S.RX_LINK_TYPE,
    (SELECT ROW_WID FROM VAR_ETL_PROC_WID) AS ETL_PROC_WID
FROM I_WC_BADGE_PRODUCT_D S
WHERE S.IND_UPDATE = 'I'

---

## Step 6: ETL Tracking Update

In [ ]:
%sql
-- Validate load results
SELECT
    'WC_BADGE_PRODUCT_D' AS TABLE_NAME,
    COUNT(*) AS TOTAL_RECORDS,
    COUNT(DISTINCT INTEGRATION_ID) AS UNIQUE_INTEGRATION_IDS,
    MAX(W_UPDATE_DT) AS LAST_UPDATE_TIME
FROM workspace.prxbi_dw_sep.WC_BADGE_PRODUCT_D
WHERE DATASOURCE_NUM_ID = 380

---

## Step 7: Cleanup - DROP TEMP VIEWs

In [ ]:
%sql
-- Drop all temporary views created during this ETL process
DROP VIEW IF EXISTS VAR_ETL_LAST_EXTRACT_TIME;
DROP VIEW IF EXISTS VAR_ETL_CURRENT_EXTRACT_TIME;
DROP VIEW IF EXISTS VAR_ETL_PROC_WID;
DROP VIEW IF EXISTS C_0FILTER;
DROP VIEW IF EXISTS I_WC_BADGE_PRODUCT_D